In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/joaovictor777/housing/housing.csv")

print(df.head())
print(df.shape)
print(df.info())

## Dimensões: 20.640 linhas e 10 colunas.

**Tipos:** 9 colunas float64 (longitude, latitude, housing_median_age, total_rooms, total_bedrooms, population, households, median_income e median_house_value) e 1 coluna categórica/object (ocean_proximity). Não há colunas int.

In [ ]:
df.describe()

- **Maior dispersão:** total_rooms (desvio-padrão ≈ 2.182), population (≈ 1.132) e total_bedrooms (≈ 421). O preço tem desvio ≈ US$ 115 mil.
- **Valores muito altos:** total_rooms chega a 39.320 e population a 35.682, enquanto as medianas são 2.127 e 1.166. Há forte assimetria à direita.
- **Valores estranhos:** o preço máximo é 500.001 e a renda máxima é 15,0001. São tetos da base. Há também quarteirões muito pequenos (total_rooms = 2, households = 1).
- **Relação com o preço:** median_income é a que mais parece se relacionar com o preço.

In [ ]:
print(df.columns)

In [ ]:
df.isnull().sum()

In [ ]:
print("Linhas duplicadas:", df.duplicated().sum())
print()
print(df.dtypes)
print()
print(df["ocean_proximity"].value_counts())
print()
print("%% de nulos em total_bedrooms: %.2f%%" % (100 * df["total_bedrooms"].isnull().mean()))

**Ausentes:** só total_bedrooms tem valores nulos (207 linhas, cerca de 1%). Decisão: preencher com a **mediana**, calculada apenas no treino. A mediana é mais robusta que a média porque a variável é assimétrica (média 538 e mediana 435). Assim nenhuma linha é descartada.

**Duplicados:** não há linhas duplicadas.

**Categórica:** ocean_proximity tem 5 categorias e foi convertida em colunas numéricas com get_dummies, pois o SVR não aceita texto.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["median_house_value"], bins=30)
plt.xlabel("Preço")
plt.ylabel("Quantidade")
plt.title("Distribuição dos preços")
plt.show()

**Gráfico 1 (histograma):** a distribuição é assimétrica à direita, concentrada entre US$ 100 mil e US$ 250 mil, com um pico no valor máximo (500.001). Esse pico é o teto da base, não um preço real.

In [ ]:
plt.boxplot(df["median_house_value"].dropna())
plt.title("Boxplot do preço")
plt.show()

num_cols = df.select_dtypes(include="number").columns
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, c in zip(axes.ravel(), num_cols):
    ax.boxplot(df[c].dropna(), vert=False)
    ax.set_title(c)
plt.tight_layout()
plt.show()

**Gráfico 2 (boxplots):** total_rooms, total_bedrooms, population e households têm muitos pontos acima do bigode superior (cauda longa). median_income tem alguns outliers superiores. Latitude, longitude e idade praticamente não têm.

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm")
plt.title("Matriz de correlação")
plt.show()

**Gráfico 3 (correlação):** a variável mais correlacionada com o preço é median_income (0,69). As demais têm correlação fraca (total_rooms 0,13; housing_median_age 0,11; latitude −0,14; population −0,03). São redundantes entre si total_rooms, total_bedrooms, population e households (correlação de 0,86 a 0,98), e também latitude e longitude (−0,92).

In [ ]:
Q1 = df["median_house_value"].quantile(0.25)
Q3 = df["median_house_value"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

In [ ]:
outliers = df[(df["median_house_value"] < limite_inferior) | (df["median_house_value"] > limite_superior)]

print("Outliers no alvo:", len(outliers))
print("Percentual: %.2f%%" % (100 * len(outliers) / len(df)))
print("Registros com preço no teto (>= 500000):", (df["median_house_value"] >= 500000).sum())

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
linhas = []
for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    qtd = ((s < li) | (s > ls)).sum()
    linhas.append({"variavel": c, "qtd_outliers": qtd, "percentual": round(100 * qtd / len(s), 2),
                   "min": s.min(), "max": s.max(), "lim_inf": round(li, 2), "lim_sup": round(ls, 2)})
tab_out = pd.DataFrame(linhas)
tab_out

**Método:** IQR (1,5 × IQR). Variáveis com outliers: total_rooms (1.287; 6,24%), total_bedrooms (1.271; 6,22%), households (1.220; 5,91%), population (1.196; 5,79%), median_income (681; 3,30%) e median_house_value (1.071; 5,19%). Longitude, latitude e idade não têm.

**Decisão:** nenhuma linha foi removida. Nas colunas de contagem usei *capping* (limites do IQR, calculados no treino), porque os valores extremos são quarteirões grandes e reais, não erros. No preço não alterei nada: os 992 registros em 500.001 são o teto da base, e removê-los eliminaria os imóveis mais caros e alteraria o conjunto de teste.

In [ ]:
X = df.drop("median_house_value", axis=1)

In [ ]:
y = df["median_house_value"]

In [ ]:
X = pd.get_dummies(X, columns=["ocean_proximity"], dtype=float)
print(X.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

In [ ]:
from sklearn.impute import SimpleImputer

imp = SimpleImputer(strategy="median").fit(X_train)
X_train_imp = pd.DataFrame(imp.transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imp.transform(X_test), columns=X.columns, index=X_test.index)
print("Nulos restantes:", X_train_imp.isnull().sum().sum(), X_test_imp.isnull().sum().sum())

In [ ]:
cols_cap = ["total_rooms", "total_bedrooms", "population", "households"]
X_train_cap, X_test_cap = X_train_imp.copy(), X_test_imp.copy()
for c in cols_cap:
    q1, q3 = X_train_cap[c].quantile(0.25), X_train_cap[c].quantile(0.75)
    li, ls = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    X_train_cap[c] = X_train_cap[c].clip(li, ls)
    X_test_cap[c] = X_test_cap[c].clip(li, ls)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train_cap)
X_train_s = scaler.transform(X_train_cap)
X_test_s = scaler.transform(X_test_cap)

In [ ]:
import time
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

EXPERIMENTOS = []

def avaliar(nome, tratamento, modelo, A_tr, A_te, ytr, yte, kernel="rbf", C=1, eps=0.1):
    t = time.time()
    modelo.fit(A_tr, ytr)
    pred = modelo.predict(A_te)
    r2 = r2_score(yte, pred)
    EXPERIMENTOS.append({"Experimento": nome, "Tratamento": tratamento, "Kernel": kernel,
                         "C": C, "Epsilon": eps, "R2 (teste)": round(r2, 4), "Tempo (s)": round(time.time() - t, 1)})
    print(f"{nome:<3} | {tratamento:<52} | R2 = {r2:.4f} ({time.time() - t:.1f}s)")
    return modelo, pred

ok_tr, ok_te = X_train.notna().all(axis=1), X_test.notna().all(axis=1)
avaliar("1", "Baseline (sem tratamento, sem escala, alvo em US$)",
        SVR(), X_train[ok_tr], X_test[ok_te], y_train_raw[ok_tr], y_test_raw[ok_te])

avaliar("2", "+ Ausentes (mediana)", SVR(), X_train_imp, X_test_imp, y_train_raw, y_test_raw)

avaliar("3", "+ Outliers (capping IQR)", SVR(), X_train_cap, X_test_cap, y_train_raw, y_test_raw)

avaliar("4", "+ Padronização (alvo ainda em US$)", SVR(), X_train_s, X_test_s, y_train_raw, y_test_raw)

Nos modelos 1 a 4 o R² ficou perto de zero (−0,05 a −0,04). No SVR, C e epsilon são medidos na unidade do alvo. Com preços na casa das centenas de milhares e epsilon = 0,1, o modelo acaba prevendo quase a média. No Modelo 5 dividi o preço por 100.000 (transformação do alvo) e o R² subiu para 0,7486. As métricas finais são convertidas de volta para dólares.

In [ ]:
y_train = y_train_raw / 1e5
y_test = y_test_raw / 1e5

m5, pred5 = avaliar("5", "+ Alvo em US$ 100 mil (C=1, eps=0.1)", SVR(), X_train_s, X_test_s, y_train, y_test)

In [ ]:
import numpy as np

rng = np.random.RandomState(42)
idx = rng.choice(len(X_train_s), size=4000, replace=False)
A_sub, y_sub = X_train_s[idx], y_train.values[idx]

In [ ]:
for k in ["linear", "rbf", "poly"]:
    avaliar("K-" + k, "Kernels (subamostra)", SVR(kernel=k), A_sub, X_test_s, y_sub, y_test, kernel=k)

In [ ]:
from sklearn.model_selection import GridSearchCV

parametros = {
    "C": [1, 3, 10, 30],
    "epsilon": [0.05, 0.1, 0.2],
    "gamma": ["scale", 0.03, 0.3],
    "kernel": ["rbf"],
}
grid = GridSearchCV(SVR(cache_size=1000), parametros, cv=3, scoring="r2", n_jobs=-1)
t = time.time()
grid.fit(A_sub, y_sub)
print("Tempo: %.0fs" % (time.time() - t))
print("Melhores parâmetros:", grid.best_params_)
print("R2 médio na validação cruzada:", round(grid.best_score_, 4))

In [ ]:
bp = grid.best_params_
modelo_final = SVR(kernel=bp["kernel"], C=bp["C"], epsilon=bp["epsilon"], gamma=bp["gamma"], cache_size=1000)
mf, pred_f = avaliar("F", "Tratados + padronizados + GridSearch (treino completo)",
                     modelo_final, X_train_s, X_test_s, y_train, y_test,
                     kernel=bp["kernel"], C=bp["C"], eps=bp["epsilon"])

**Kernels** (mesma subamostra de 4.000 linhas): RBF 0,7192 > poly 0,6597 > linear 0,6199. O RBF captura relações não lineares que o linear não captura.

**GridSearch** (cv = 3, subamostra de 4.000 linhas, por causa do custo do SVR): melhores parâmetros C = 30, epsilon = 0,2, gamma = "scale", kernel RBF (R² médio de validação 0,7549). O modelo final foi treinado com todo o treino.

In [ ]:
pred_usd = pred_f * 1e5
r2 = r2_score(y_test_raw, pred_usd)
mae = mean_absolute_error(y_test_raw, pred_usd)
rmse = np.sqrt(mean_squared_error(y_test_raw, pred_usd))
print("R2   : %.4f" % r2)
print("MAE  : US$ %.0f" % mae)
print("RMSE : US$ %.0f" % rmse)
print("Meta R2 > 0,65:", "ATINGIDA" if r2 > 0.65 else "NÃO atingida")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test_raw, pred_usd, s=6, alpha=0.35)
lim = [0, max(y_test_raw.max(), pred_usd.max())]
plt.plot(lim, lim, "r--", label="Predição perfeita")
plt.xlabel("Valor real (US$)")
plt.ylabel("Valor previsto (US$)")
plt.title("Real x Previsto (teste)")
plt.legend()
plt.show()

In [ ]:
pd.DataFrame(EXPERIMENTOS)

Os pontos seguem a diagonal, então as previsões ficam próximas dos valores reais, com mais dispersão nos preços altos. A coluna vertical em 500.001 são imóveis com preço cortado no teto da base, que o modelo tende a subestimar.

- **Atingiu R² > 0,65?** Sim, R² = 0,7737 no teste (MAE US$ 36.172, RMSE US$ 54.453).
- **Maior impacto:** a transformação do alvo (R² de −0,04 para 0,7486).
- **Outliers melhoraram?** Praticamente não: do Modelo 2 para o 3 o R² foi de −0,0487 para −0,0482. O modelo ainda estava limitado pela escala do alvo.
- **Melhor kernel:** RBF (0,7192).
- **Hiperparâmetros finais:** RBF, C = 30, epsilon = 0,2, gamma = "scale".
- **Dificuldades:** o SVR é lento com muitas linhas (usei subamostra e grade reduzida); o R² ficou perto de zero até reescalar o alvo; o teto de 500.001 no preço.
- **Como melhorar:** grade maior com C até 1000, tratar o teto do preço, usar log do alvo, criar atributos (como cômodos por domicílio) e comparar com Random Forest ou Gradient Boosting.